## Импорт библиотек

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import io
import time
import json
import shutil
import zipfile
import requests

import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.ticker import MaxNLocator
from rich.progress import track
from pathlib import Path

from scipy.signal import find_peaks
from scipy.stats import kurtosis, skew, pearsonr
from scipy.signal.windows import blackman

from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.preprocessing import PowerTransformer, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, StackingClassifier, VotingClassifier


## Создание названий признаков

**FD - частотная область, TD - временна́я область**

In [3]:
# Список для хранения названий всех признаков
cols = []

# Базовые статистические признаки
common_cols = ["Median", "Mean", "Std", "Ptp", "Max", "Min", "P90", "P75", "P25", 
               "L1", "Kurtosis", "Skewness", "Integral", "RMS", "IntegralSq", 
               "SumSq", "CumsumFinal", "CumsumMax", "CumsumMin", "CumsumRange", 
               "Variation", "OutlierCount", "PeakCount"]

# Спектральные признаки
frequency_cols = ["EnergySpectralCentroid", "EnergySpectralBandwidth", 
                  "MeanAmpAbove10Hz", "StdAmpAbove10Hz", "EnergyAbove10Hz"]

# Признаки для магнитуд гироскопа и акселерометра
for channel in ["mag_g", "mag_a"]:
    for dif in ["", "dif_", "dif_dif_"]:
        for col_name in common_cols:
            cols.append(f"{channel}_TD_{dif}{col_name}")

# Признаки для каждой оси отдельно
for channel in ["gx", "gy", "gz", "ax", "ay", "az"]:
    for dif in ["", "dif_", "dif_dif_"]:
        for col_name in common_cols:
            cols.append(f"{channel}_TD_{dif}{col_name}")
        for col_name in common_cols + frequency_cols:
            cols.append(f"{channel}_FD_{dif}{col_name}")

# Подсчет общего количества признаков
num_of_features = len(cols)

# Добавление целевой переменной и создание пустого DataFrame
cols.append("Класс")
features_df_orig = pd.DataFrame(columns=cols)

print(f"Количество признаков: {num_of_features}")


Количество признаков: 1056


## Извлечение признаков

In [4]:
def feature_extraction(seg, features, frequencies=None, more10hz_idx=None):
    
    # Предварительные вычисления
    square_seg = seg**2
    sum_square_seg = np.sum(square_seg)
    
    std_seg = np.std(seg)
    mean_seg = np.mean(seg)
    median_seg = np.median(seg)

    cumsum = np.cumsum(seg)
    max_cumsum = np.max(cumsum)
    min_cumsum = np.min(cumsum)

    # Базовые статистики
    features.append(median_seg)              # Медиана
    features.append(mean_seg)                # Средняя амплитуда
    features.append(std_seg)                 # Стандартное отклонение
    features.append(np.ptp(seg))             # Размах
    features.append(np.max(seg))             # Максимальная амплитуда
    features.append(np.min(seg))             # Минимальная амплитуда
    features.append(np.percentile(seg, 90))  # 90-й процентиль
    features.append(np.percentile(seg, 75))  # 75-й процентиль
    features.append(np.percentile(seg, 25))  # 25-й процентиль
    features.append(np.sum(np.abs(seg)))     # Сумма модулей
    
    # Форма распределения
    features.append(kurtosis(seg))  # Куртозис
    features.append(skew(seg))      # Ассиметрия

    # Энергетические характеристики
    features.append(np.trapz(seg))                 # Интеграл
    features.append(np.sqrt(np.mean(square_seg)))  # RMS    
    features.append(np.trapz(square_seg))          # Интеграл (квадрат)
    features.append(sum_square_seg)                # Сумма квадратов

    # Кумулятивные характеристики
    features.append(cumsum[-1])               # Чистый дрейф за окно
    features.append(max_cumsum)               # Максимум накопления
    features.append(min_cumsum)               # Минимум накопления  
    features.append(max_cumsum - min_cumsum)  # Размах дрейфа
    
    # Коэффициент вариации
    features.append(std_seg / mean_seg if mean_seg != 0 else 0)  
    
    # Количество выбросов
    features.append(np.sum(np.abs(seg - median_seg) > 0.5 * std_seg))

    # Количество пиков
    features.append(len(find_peaks(seg, height=median_seg + 0.5 * std_seg)[0]))

    # Спектральные моменты
    if frequencies is not None:
        # Центр тяжести спектра
        centroid = np.sum(frequencies * square_seg) / sum_square_seg
        features.append(centroid) 
        # Ширина
        features.append(np.sqrt(np.sum((frequencies - centroid)**2 * square_seg) / sum_square_seg))  

    # Исследование частот >10 Гц
    if more10hz_idx is not None:
        amp_half_10 = seg[more10hz_idx:]
        features.append(np.mean(amp_half_10))        # Средняя амплитуда частот >10 Гц
        features.append(np.std(amp_half_10))         # Стандартное отклонение частот >10 Гц
        features.append(np.trapz(amp_half_10 ** 2))  # Энергия частот >10 Гц (квадрат)
        
    return features


In [ ]:
Fs = 195                         # Частота дискретизации данных (Гц)
overlaps = [50, 75]             # Перекрытия окон (%)
window_sizes = [8, 16, 32, 64, 128, 256]  # Размеры окон
window_sizes = window_sizes[::-1]

data_dir = 'ProcessedSortedDataset/'  # Путь к папкам с данными
class_folders = {
    0: '0', 
    1: '1', 
    2: '2', 
    3: '3', 
    4: '4'
}

# Цикл по размерам окон
for window_size in window_sizes:
    print("ОКНО", window_size)
    
    window = blackman(window_size)                  # Окно Блэкмана-Наталла для уменьшения утечки спектра
    freqs_0 = np.fft.rfftfreq(window_size, 1 / Fs)  # Получение только положительных частот для спектра
    more10hz_idx_0 = np.sum(freqs_0 < 10)           # Индекс для частот более 10 Гц

    freqs_1 = (freqs_0[:-1] + freqs_0[1:]) / 2      # Для diff_amp_half
    more10hz_idx_1 = np.sum(freqs_1 < 10)           # Индекс для частот более 10 Гц
    
    freqs_2 = (freqs_1[:-1] + freqs_1[1:]) / 2      # Для diff_diff_amp_half
    more10hz_idx_2 = np.sum(freqs_2 < 10)           # Индекс для частот более 10 Гц

    strides = [int(window_size * (1 - overlap / 100)) for overlap in overlaps]
    
    # Цикл по величинам перекрытий
    for stride in strides:  # Цикл по перекрытиям 50% и 75%      
        rows = []
        
        # Цикл по папкам классов
        for class_label, folder_name in class_folders.items():     
            print(class_label)
            folder_path = os.path.join(data_dir, folder_name)   
            file_names = [name for name in os.listdir(folder_path) if name.endswith('.csv')]

            # Цикл по файлам в папке определенного класса
            for file_name in file_names:
                file_path = os.path.join(folder_path, file_name)
        
                # Загрузка данных из CSV-файла и извлечение значений гироскопа и акселерометра
                df = pd.read_csv(file_path)
                data = df[['gx', 'gy', 'gz', 'ax', 'ay', 'az']].values  
                
                # Скользящее окно по даннымс заданным шагом
                for start in range(0, len(data) - window_size + 1, stride):
                    segment = data[start:start + window_size]
                    segment_windowed = segment * window[:, None]  # Наложение окна
                    
                    yf = np.fft.rfft(segment_windowed, axis=0)    # Применение БПФ
                    amplitude = np.abs(yf)                        # Извлечение амплитуд

                    features = []
        
                    # Извлечение признаков из магнитуд гироскопа и акселерометра, их производных и производных их производных
                    mag_g = np.sqrt(segment[:, 0]**2 + segment[:, 1]**2 + segment[:, 2]**2)
                    mag_a = np.sqrt(segment[:, 3]**2 + segment[:, 4]**2 + segment[:, 5]**2)
        
                    # Первые производные магнитуд
                    diff_mag_g = np.diff(mag_g)
                    diff_diff_mag_g = np.diff(diff_mag_g)
        
                    # Вторые производные магнитуд
                    diff_mag_a = np.diff(mag_a)
                    diff_diff_mag_a = np.diff(diff_mag_a)
                            
                    # Извлечение временных признаков из всех данных магнитуд
                    for seg in [mag_g, diff_mag_g, diff_diff_mag_g, mag_a, diff_mag_a, diff_diff_mag_a]:
                        features = feature_extraction(seg, features)

                    # Извлечение признаков из каждой оси гироскопа и акселерометра, их производных и производных их производных
                    for channel in range(amplitude.shape[1]):
                        amp_half = amplitude[:, channel] 
                        
                        diff_amp_half = np.diff(amp_half)            # Первая производная амплитуд спектра
                        diff_diff_amp_half = np.diff(diff_amp_half)  # Вторая производная амплитуд спектра
                        
                        seg = segment[:, channel]                     
                        diff_seg = np.diff(seg)                      # Первая производная временных данных
                        diff_diff_seg = np.diff(diff_seg)            # Вторая производная временных данных
        
                        # Обработка исходных данных и их производных
                        for i, (amp_half_i, seg_i, freqs, more10hz_idx) in enumerate([[amp_half, seg, freqs_0, more10hz_idx_0], 
                                                                                      [diff_amp_half, diff_seg, freqs_1, more10hz_idx_1], 
                                                                                      [diff_diff_amp_half, diff_diff_seg, freqs_2, more10hz_idx_2]]):
                            
                            features = feature_extraction(seg_i, features)                            # Извлечение временных признаков
                            features = feature_extraction(amp_half_i, features, freqs, more10hz_idx)  # Извлечение частотных признаков
                    
                    # Добавляем метку класса и сохраняем извлеченные признаки
                    features.append(class_label)
                    rows.append(features)

        features_df = pd.DataFrame(rows, columns=cols)
        features_df = features_df.fillna(0)
        features_df.to_csv(f"DATA/features/REAL_features_ws{window_size}_overlap{int((1 - stride / window_size) * 100)}.csv", index=False)   
        print(f"Строк данных сохранено: {len(features_df)}")


ОКНО 256
0


## Обучение моделей

In [16]:
# Конвертер numpy типов для JSON
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        
        if isinstance(obj, np.integer):
            return int(obj)
            
        if isinstance(obj, np.floating):
            return float(obj)
            
        if isinstance(obj, np.ndarray):
            return obj.tolist()
            
        return super().default(obj)


def prepare_data(features_df, test_size=0.2, scaler_type=None, random_state=42):
    
    # Разбиение данных на признаки и целевую переменную
    X = features_df.drop(columns=["Класс"])
    y = features_df["Класс"].values

    # Разделение на обучающую и тестовую выборки
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

    # Для SVM, KNN, Logistic Regression 
    if scaler_type == "standard":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # Для нейросетей
    elif scaler_type == "minmax":
        scaler = MinMaxScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # Если есть выбросы
    elif scaler_type == "robust":
        scaler = RobustScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    # Если данные не нормально распределены (скошенные)
    elif scaler_type == "power":
        power = PowerTransformer(method="yeo-johnson")
        X_train = power.fit_transform(X_train)
        X_test = power.transform(X_test)

    return X_train, X_test, y_train, y_test


def train_and_evaluate(models, all_results, results_key, X_train_raw, X_test_raw, X_train_std, X_test_std, y_train, y_test):
    
    trained_models = {}
    ensamble_models = ["Histogram-based Gradient Boosting", "XGBoost", "LightGBM"]
    
    for model_name, model in models.items():
        
        # Выбор подходящих данных для конкретной модели
        if model_name in ["K-Nearest Neighbors", "Logistic Regression"]:
            X_train, X_test = X_train_std, X_test_std
        else:
            X_train, X_test = X_train_raw, X_test_raw

        print("-" * 53)
        print(f"\n{model_name}")
        
        # Обучение
        start_train = time.perf_counter()
        model.fit(X_train, y_train)
        training_time = time.perf_counter() - start_train
        
        # Сохраняем обученную модель, если она используется в Stacking и Voting
        if model_name in ensamble_models:
            trained_models[model_name] = model
            print(model_name)
        
        # Инференс
        start_predict = time.perf_counter()
        y_pred = model.predict(X_test)
        predict_time = time.perf_counter() - start_predict

        # Расчет метрик качества
        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, average="weighted") * 100
        recall = recall_score(y_test, y_pred, average="weighted") * 100
        f1 = f1_score(y_test, y_pred, average="weighted") * 100

        # Время на одно предсказание (в миллисекундах)
        time_per_pred = predict_time / len(y_test) * 1000

        # Сохранение результатов
        all_results[results_key][model_name] = {
            "Accuracy": round(accuracy, 3),
            "Precision": round(precision, 3),
            "Recall": round(recall, 3),
            "F1_Score": round(f1, 3),
            "Training_Time": round(training_time, 2),
            "Inference_Time": round(predict_time, 4),
            "Time_per_pred_ms": round(time_per_pred, 4)
        }

        all_results[results_key][model_name]["Confusion_Matrix"] = confusion_matrix(y_test, y_pred).tolist()
        
        # Вывод результатов
        print(f"Время обучения: {training_time:.3f} с")
        print(f"Время инференса: {predict_time:.3f} с, {time_per_pred:.4f} мс на 1 предсказание\n")
        print(classification_report(y_test, y_pred))
        print(f"Accuracy:\t{accuracy:.2f} %")
        
    return trained_models, all_results


In [17]:
overlaps = [50, 75]             # Перекрытия окон (%)
window_sizes = [64, 32, 16, 8]  # Размеры окон

# Получение списка CSV-файлов с извлеченными признаками
file_names = [name for name in os.listdir("DATA/features") 
              if name.endswith('.csv') and
              any(f"ws{w}" in name for w in window_sizes) and
              any(f"overlap{o}" in name for o in overlaps)]

# Сортировка сначала по убыванию размера окна, затем по возрастанию перекрытия
file_names = sorted(
    file_names,
    key=lambda x: (-int(x.split('_')[2].replace('ws', '')), 
                   int(x.split('_')[3].replace('overlap', '').replace('.csv', '')))
)

all_results = {}   # Словарь для всех результатов
random_state = 42  # Фиксируем random_state для воспроизводимости
file_names = ["REAL_features_ws256_overlap75.csv"]
selected_features = ['gy_TD_dif_dif_Min', 'gz_TD_dif_Min', 'ax_TD_Min', 'gz_TD_dif_dif_Min', 'ax_TD_dif_dif_Min', 'ay_TD_Min', 'ax_TD_dif_Min', 'gy_TD_Min', 'az_TD_Min', 'gy_TD_dif_Min', 'gx_TD_dif_Min', 'ay_TD_dif_Min', 'az_TD_dif_dif_Max', 'ay_TD_dif_dif_Max', 'ax_TD_dif_dif_Max', 'gz_TD_dif_dif_Max', 'gz_TD_dif_Max', 'gz_TD_Max', 'ax_TD_Max', 'ay_TD_Max', 'gy_TD_dif_dif_Max', 'az_TD_Max', 'ax_TD_dif_Max', 'gy_TD_dif_Max', 'gx_TD_dif_Max', 'gy_TD_Max', 'az_TD_dif_Max', 'gx_TD_Min', 'gy_TD_dif_dif_CumsumFinal', 'gz_TD_L1', 'az_TD_L1', 'gz_TD_dif_dif_L1', 'gx_TD_dif_L1', 'gy_TD_dif_dif_L1', 'ax_TD_dif_dif_L1', 'ax_TD_L1', 'ay_TD_dif_L1', 'ay_TD_L1', 'gz_TD_dif_L1', 'az_TD_dif_dif_L1', 'ay_TD_dif_dif_L1', 'az_TD_dif_Mean', 'gy_TD_Mean', 'gx_TD_Ptp', 'gx_TD_dif_Ptp', 'az_TD_dif_dif_Ptp', 'gz_TD_dif_Ptp', 'ay_TD_dif_Ptp', 'ax_TD_dif_dif_Ptp', 'az_TD_dif_Ptp', "Класс"]

for file_name in file_names:
    print(f"\n\n{'=' * 53}")
    print(f"ОБРАБОТКА ФАЙЛА: {file_name}")
    print('=' * 53)

    # Загрузка данных
    features_df = pd.read_csv(f"DATA/features/{file_name}")
    features_df = features_df[selected_features]
    print("\nДанные загружены")

    # Подготовка данных без масштабирования
    X_train_raw, X_test_raw, y_train, y_test = prepare_data(features_df, test_size=0.2, scaler_type=None, random_state=random_state)
    
    # Подготовка данных с использованием StandardScaler 
    X_train_std, X_test_std, _, _ = prepare_data(features_df, test_size=0.2, scaler_type="standard", random_state=random_state)
    print("Предобработка и разбиение на train/test завершены\n")

    # Инициализация базовых моделей
    models = {
        "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=37, leaf_size=1, p=1, metric='manhattan', weights='uniform'),
        "Logistic Regression": LogisticRegression(random_state=random_state),
        "Naive Bayes": GaussianNB(var_smoothing=1e-5),
        "Decision Tree": DecisionTreeClassifier(max_depth=30, criterion='entropy', random_state=random_state),
        "CatBoost": CatBoostClassifier(silent=True, random_state=random_state, task_type="GPU", devices="0:1:2:3", allow_writing_files=False, save_snapshot=False),
        "Histogram-based Gradient Boosting": HistGradientBoostingClassifier(random_state=random_state),
        "XGBoost": XGBClassifier(random_state=random_state, device="cuda", tree_method="hist", n_jobs=-1),
        "LightGBM": LGBMClassifier(verbose=-1, random_state=random_state)
    }

    # Инициализация хранилища результатов
    results_key = f"{file_name}"
    all_results[results_key] = {}

    # Обучение базовых моделей
    trained_models, all_results = train_and_evaluate(models, all_results, results_key, X_train_raw, X_test_raw, X_train_std, X_test_std, y_train, y_test)

    # Создание ансамблей на основе сохраненных моделей
    new_models = {
        "Stacking Classifier": StackingClassifier(
            estimators=[("LightGBM", trained_models["LightGBM"]),
                        ("XGBoost", trained_models["XGBoost"]),
                        ("Histogram-based Gradient Boosting", trained_models["Histogram-based Gradient Boosting"])],
            final_estimator=LogisticRegression(random_state=random_state)),
        
        "Voting Classifier": VotingClassifier(
            estimators=[("LightGBM", trained_models["LightGBM"]),
                        ("XGBoost",trained_models["XGBoost"]),
                        ("Histogram-based Gradient Boosting", trained_models["Histogram-based Gradient Boosting"])],
            voting='soft')
    }

    # Обучение ансамблей
    _, all_results = train_and_evaluate(new_models, all_results, results_key, X_train_raw, X_test_raw, X_train_std, X_test_std, y_train, y_test)

# Сохранение результатов обучения в json файл
with open('DATA/related_info/models_training_results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=4, cls=NumpyEncoder, ensure_ascii=False)
print(f"\nРезультаты сохранены в json файл")




ОБРАБОТКА ФАЙЛА: REAL_features_ws256_overlap75.csv

Данные загружены
Предобработка и разбиение на train/test завершены

-----------------------------------------------------

CatBoost
Время обучения: 18.212 с
Время инференса: 0.028 с, 0.0051 мс на 1 предсказание

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1037
           1       0.99      0.99      0.99      1145
           2       0.97      0.97      0.97      1113
           3       0.96      0.96      0.96      1125
           4       0.97      0.96      0.97      1108

    accuracy                           0.98      5528
   macro avg       0.98      0.98      0.98      5528
weighted avg       0.98      0.98      0.98      5528

Accuracy:	97.65 %
-----------------------------------------------------

Histogram-based Gradient Boosting
Histogram-based Gradient Boosting
Время обучения: 21.482 с
Время инференса: 0.041 с, 0.0074 мс на 1 предсказание

              precision 

## Сравнение моделей

In [18]:
# Настройка стилей для графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Загрузка результатов
with open('DATA/related_info/models_training_results.json', 'r', encoding='utf-8') as f:
    all_results = json.load(f)

print(f"Загружено результатов для {len(all_results)} конфигураций")


Загружено результатов для 1 конфигураций


In [19]:
# Создаем DataFrame для всех результатов
data = []
for file_name, models_results in all_results.items():
    # Извлекаем ws и overlap из имени файла
    ws = int(file_name.split('ws')[1].split('_')[0])
    overlap = int(file_name.split('overlap')[1].split('.')[0])
    
    for model_name, metrics in models_results.items():
        data.append({
            'Window Size': ws,
            'Overlap': overlap,
            'Model': model_name,
            'Accuracy': metrics['Accuracy'],
            'Precision': metrics['Precision'],
            'Recall': metrics['Recall'],
            'F1_Score': metrics['F1_Score'],
            'Training_Time': metrics['Training_Time'],
            'Inference_Time': metrics['Inference_Time'],
            'Time_per_pred_ms': metrics['Time_per_pred_ms']
        })

df_results = pd.DataFrame(data)
df_results


,Window Size,Overlap,Model,Accuracy,Precision,Recall,F1_Score,Training_Time,Inference_Time,Time_per_pred_ms
0,256,75,CatBoost,97.648,97.648,97.648,97.647,18.21,0.0282,0.0051
1,256,75,Histogram-based Gradient Boosting,98.499,98.498,98.499,98.498,21.48,0.0406,0.0074
2,256,75,XGBoost,98.064,98.065,98.064,98.064,5.91,0.1223,0.0221
3,256,75,LightGBM,98.281,98.281,98.281,98.280,8.06,0.0416,0.0075
4,256,75,Stacking Classifier,98.480,98.478,98.480,98.479,202.67,0.2883,0.0522
5,256,75,Voting Classifier,98.426,98.424,98.426,98.425,37.57,0.2845,0.0515
